# Named Entity Recognition (NER)

## Learning Objectives
1. Implement BIO tagging from scratch — encoding entity spans as token-level labels
2. Build a BiLSTM sequence tagger in PyTorch trained with cross-entropy loss on NER data
3. Apply Viterbi decoding with transition constraints to improve boundary detection
4. Evaluate at span level (exact match) and contrast with token-level F1 metrics

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: BIO Tagging in NumPy

Named Entity Recognition assigns one of seven tags to each token:
`O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC`

The **BIO scheme** encodes span boundaries:
- `B-<type>` — first token of an entity of that type
- `I-<type>` — continuation token of the same entity
- `O` — outside any entity

We implement `span_to_bio` and `bio_to_spans` and verify the round-trip property.

In [ ]:
# Level 1: BIO tagging utilities

TAG2ID = {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4,
          'B-LOC': 5, 'I-LOC': 6}
ID2TAG = {v: k for k, v in TAG2ID.items()}
N_TAGS = len(TAG2ID)

def span_to_bio(tokens: list, spans: list) -> list:
    """
    Convert entity spans to a BIO tag sequence.

    Args:
        tokens: list of string tokens, e.g. ['John', 'Smith', 'works', 'at', 'Google']
        spans:  list of (start, end_exclusive, entity_type) tuples
                e.g. [(0, 2, 'PER'), (4, 5, 'ORG')]

    Returns:
        list of BIO tag strings, one per token
    """
    tags = ['O'] * len(tokens)
    for start, end, etype in spans:
        for i in range(start, end):
            if i == start:
                tags[i] = f'B-{etype}'
            else:
                tags[i] = f'I-{etype}'
    return tags


def bio_to_spans(tokens: list, tags: list) -> list:
    """
    Recover entity spans from a BIO tag sequence.

    Args:
        tokens: list of string tokens
        tags:   list of BIO tag strings

    Returns:
        list of (start, end_exclusive, entity_type, text) tuples
    """
    spans = []
    i = 0
    while i < len(tags):
        tag = tags[i]
        if tag.startswith('B-'):
            etype = tag[2:]
            start = i
            j = i + 1
            while j < len(tags) and tags[j] == f'I-{etype}':
                j += 1
            surface = ' '.join(tokens[start:j])
            spans.append((start, j, etype, surface))
            i = j
        else:
            i += 1
    return spans


# ── Demonstrate on 10 synthetic sentences ───────────────────────────────────
PERSON_NAMES = ['Alice', 'Bob', 'Carol', 'David', 'Eve']
ORG_NAMES    = ['Google', 'OpenAI', 'Microsoft', 'Meta', 'Apple']
LOC_NAMES    = ['Paris', 'London', 'Berlin', 'Tokyo', 'Sydney']
FILLER_TOKS  = ['works', 'at', 'visited', 'from', 'in', 'leads', 'joined', 'left']

rng = np.random.default_rng(7)

def make_synthetic_sentence() -> tuple:
    """Return (tokens, spans) with 1-2 entities per sentence."""
    name  = rng.choice(PERSON_NAMES)
    org   = rng.choice(ORG_NAMES)
    loc   = rng.choice(LOC_NAMES)
    fname = name.split()[0]  # single-token name for simplicity
    mid   = rng.choice(FILLER_TOKS)
    sentence = [fname, mid, org, 'in', loc]
    spans = [(0, 1, 'PER'), (2, 3, 'ORG'), (4, 5, 'LOC')]
    return sentence, spans

test_cases = [make_synthetic_sentence() for _ in range(10)]
print(f"{'Tokens':<35}  {'BIO Tags':<45}  {'Recovered Spans'}")
print('-' * 120)
all_rt_ok = True
for tokens, gold_spans in test_cases:
    bio = span_to_bio(tokens, gold_spans)
    recovered = bio_to_spans(tokens, bio)
    # Round-trip check: recovered spans (start, end, type) match gold_spans
    rec_set  = {(s, e, t) for s, e, t, _ in recovered}
    gold_set = {(s, e, t) for s, e, t in gold_spans}
    rt_ok = rec_set == gold_set
    all_rt_ok = all_rt_ok and rt_ok
    ent_strs = [f"{t}:{s}-{e}" for s, e, t, _ in recovered]
    print(f"{str(tokens):<35}  {str(bio):<45}  {ent_strs}  RT={'OK' if rt_ok else 'FAIL'}")

print(f"\nRound-trip verification (all 10 sentences): {'PASS' if all_rt_ok else 'FAIL'}")

## Level 2: BiLSTM NER Tagger

Bidirectional LSTMs read sequences left-to-right AND right-to-left, giving each
token access to full context. This is particularly important for NER because the
entity type of a token often depends on what follows it.

We train on a synthetic NER dataset of 200 sentences (7 tags × token cross-entropy)
with padding masks to ignore loss on pad positions.

In [ ]:
# Level 2: BiLSTM NER tagger

# ── Synthetic NER dataset ────────────────────────────────────────────────────
MAX_LEN = 12
PAD_ID  = 0

def build_ner_vocab() -> dict:
    all_words = (PERSON_NAMES + ORG_NAMES + LOC_NAMES
                 + list(FILLER_TOKS) + ['the', 'a', 'and', 'of', '.'])
    return {w: i+1 for i, w in enumerate(sorted(set(all_words)))}

ner_vocab = build_ner_vocab()
NER_VOCAB_SZ = len(ner_vocab) + 1


def make_ner_sample(max_len: int = MAX_LEN) -> tuple:
    """
    Generate a padded token-id sequence and corresponding tag-id sequence.
    Returns (token_ids, tag_ids) both length max_len; pad positions = -100 for tags.
    """
    tokens, spans = make_synthetic_sentence()
    tags  = span_to_bio(tokens, spans)
    ids   = [ner_vocab.get(t, 0) for t in tokens]
    t_ids = [TAG2ID[tg] for tg in tags]
    # Pad to max_len
    pad_len = max_len - len(ids)
    ids   = ids   + [PAD_ID] * pad_len
    t_ids = t_ids + [-100]   * pad_len   # -100 = ignore in loss
    return ids, t_ids


dataset = [make_ner_sample() for _ in range(200)]
X_ner = torch.tensor([d[0] for d in dataset], dtype=torch.long)
y_ner = torch.tensor([d[1] for d in dataset], dtype=torch.long)

# 80/20 train/test
n_ner = len(X_ner)
X_ner_tr, X_ner_te = X_ner[:160], X_ner[160:]
y_ner_tr, y_ner_te = y_ner[:160], y_ner[160:]
print(f"NER train: {len(X_ner_tr)}  test: {len(X_ner_te)}")


class BiLSTMNER(nn.Module):
    """Bidirectional LSTM sequence tagger for NER."""
    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, num_tags: int):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.bilstm  = nn.LSTM(embed_dim, hidden_dim,
                                batch_first=True, bidirectional=True)
        self.fc      = nn.Linear(hidden_dim * 2, num_tags)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: [B, T] padded token ids
        Returns:
            logits: [B, T, num_tags]
        """
        emb = self.embed(x)
        out, _ = self.bilstm(emb)   # [B, T, 2*H]
        return self.fc(out)          # [B, T, num_tags]


bilstm_ner = BiLSTMNER(NER_VOCAB_SZ, embed_dim=32, hidden_dim=64,
                        num_tags=N_TAGS).to(device)
opt_ner   = optim.Adam(bilstm_ner.parameters(), lr=1e-3)
ner_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

train_losses = []
for epoch in range(120):
    bilstm_ner.train()
    logits_ner = bilstm_ner(X_ner_tr.to(device))         # [B, T, 7]
    loss_ner   = ner_loss_fn(
        logits_ner.view(-1, N_TAGS),
        y_ner_tr.to(device).view(-1))
    opt_ner.zero_grad(); loss_ner.backward(); opt_ner.step()
    train_losses.append(loss_ner.item())

print(f"Training complete. Initial loss: {train_losses[0]:.4f}  Final: {train_losses[-1]:.4f}")


def greedy_decode(logits: torch.Tensor) -> np.ndarray:
    """Greedy argmax decoding. Returns [B, T] tag ids."""
    return logits.argmax(dim=-1).cpu().numpy()


bilstm_ner.eval()
with torch.no_grad():
    test_logits = bilstm_ner(X_ner_te.to(device))

greedy_preds = greedy_decode(test_logits)

# Token-level accuracy (ignoring pad positions)
mask_te = (y_ner_te != -100).numpy()
correct = (greedy_preds[mask_te] == y_ner_te.numpy()[mask_te]).sum()
total   = mask_te.sum()
print(f"Greedy token-level accuracy: {correct/total:.3f}  ({correct}/{total} tokens)")

## Real-World Example 1: Viterbi Decoding with Transition Constraints

Greedy decoding ignores sequence structure: it can produce invalid BIO sequences
like `O → I-PER` (continuation without a preceding B-tag).

**Viterbi decoding** finds the globally most probable tag sequence subject to
hard transition constraints encoded in a score matrix:
- `I-PER` cannot directly follow `B-ORG` (type switch mid-entity)
- `I-PER` cannot start a sentence (must be preceded by `B-PER`)

We build a simple Viterbi decoder and compare span-level recovery against greedy.

In [ ]:
# Real-World Example 1: Viterbi decoding with transition constraints

def build_transition_matrix(n_tags: int, tag2id: dict) -> np.ndarray:
    """
    Build a log-probability transition matrix where invalid transitions get -inf.

    Valid:  O->O, O->B-*, B-X->I-X, B-X->O, B-X->B-Y, I-X->I-X, I-X->O, I-X->B-Y
    Invalid: O->I-* (continuation without B), B-X->I-Y (type switch)
    """
    mat = np.zeros((n_tags, n_tags), dtype=np.float32)
    INF = -1e9
    for from_tag, from_id in tag2id.items():
        for to_tag, to_id in tag2id.items():
            # Disallow: I-X following anything other than B-X or I-X
            if to_tag.startswith('I-'):
                to_type = to_tag[2:]
                if not (from_tag == f'B-{to_type}' or from_tag == f'I-{to_type}'):
                    mat[from_id, to_id] = INF
    return mat

transition = build_transition_matrix(N_TAGS, TAG2ID)
print("Transition constraints (invalid transitions = -1e9):")
print(f"  O -> I-PER: {transition[TAG2ID['O'], TAG2ID['I-PER']]:.0f}  (invalid)")
print(f"  B-PER -> I-PER: {transition[TAG2ID['B-PER'], TAG2ID['I-PER']]:.0f}  (valid)")
print(f"  B-ORG -> I-PER: {transition[TAG2ID['B-ORG'], TAG2ID['I-PER']]:.0f}  (invalid)")


def viterbi_decode(emission_logits: np.ndarray,
                   transition_mat: np.ndarray) -> np.ndarray:
    """
    Viterbi decoding over a sequence with a fixed transition matrix.

    Args:
        emission_logits: [T, num_tags] log-probabilities (or logits) per position
        transition_mat:  [num_tags, num_tags] where mat[i,j] = score of i->j

    Returns:
        tag_ids: [T] best tag sequence
    """
    T, K = emission_logits.shape
    viterbi = np.full((T, K), -1e18, dtype=np.float32)
    backptr = np.zeros((T, K), dtype=np.int32)

    # Initialise from first token
    viterbi[0] = emission_logits[0]

    for t in range(1, T):
        for tag in range(K):
            scores = viterbi[t-1] + transition_mat[:, tag] + emission_logits[t, tag]
            backptr[t, tag] = scores.argmax()
            viterbi[t, tag] = scores.max()

    # Back-trace
    sequence = np.zeros(T, dtype=np.int32)
    sequence[-1] = viterbi[-1].argmax()
    for t in range(T-2, -1, -1):
        sequence[t] = backptr[t+1, sequence[t+1]]
    return sequence


def decode_sequence(model: BiLSTMNER, token_ids: torch.Tensor,
                    use_viterbi: bool = False,
                    trans: np.ndarray = None) -> list:
    """Run model on a single token sequence and return predicted tags."""
    model.eval()
    with torch.no_grad():
        logits = model(token_ids.unsqueeze(0).to(device)).squeeze(0).cpu().numpy()
    if use_viterbi:
        return viterbi_decode(logits, trans).tolist()
    return logits.argmax(-1).tolist()


# Compare greedy vs viterbi on test set: count valid BIO sequences
def count_invalid_transitions(tag_sequence: list) -> int:
    """Count transitions that violate BIO constraints."""
    n_invalid = 0
    for i in range(1, len(tag_sequence)):
        t_prev = ID2TAG[tag_sequence[i-1]]
        t_curr = ID2TAG[tag_sequence[i]]
        if t_curr.startswith('I-'):
            curr_type = t_curr[2:]
            if not (t_prev == f'B-{curr_type}' or t_prev == f'I-{curr_type}'):
                n_invalid += 1
    return n_invalid

greedy_invalid = 0
viterbi_invalid = 0
for i in range(len(X_ner_te)):
    seq = X_ner_te[i]
    # Only count non-pad positions
    actual_len = (y_ner_te[i] != -100).sum().item()
    greedy_tags  = decode_sequence(bilstm_ner, seq, use_viterbi=False)[:actual_len]
    viterbi_tags = decode_sequence(bilstm_ner, seq, use_viterbi=True, trans=transition)[:actual_len]
    greedy_invalid  += count_invalid_transitions(greedy_tags)
    viterbi_invalid += count_invalid_transitions(viterbi_tags)

print(f"\nInvalid BIO transitions — Greedy: {greedy_invalid}  |  Viterbi: {viterbi_invalid}")
print("Viterbi eliminates all constraint violations by construction.")

## Real-World Example 2: Span-Level F1 Evaluation

Token-level accuracy counts each token independently. But for NER the meaningful
unit is the **entity span** — a prediction is only correct if both boundaries and
the entity type are exactly right.

Span-level metrics are typically 5-15% lower than token-level accuracy because a
single boundary error (off by one token) counts as two token mistakes but one
full span error.

In [ ]:
# Real-World Example 2: Span-level F1 evaluation

def extract_spans_from_ids(tag_ids: list, actual_len: int) -> set:
    """
    Convert a tag-id sequence to a set of (start, end, type) span tuples.
    Only considers positions 0..actual_len-1.
    """
    tag_strs = [ID2TAG[t] for t in tag_ids[:actual_len]]
    fake_tokens = [str(i) for i in range(actual_len)]
    spans = bio_to_spans(fake_tokens, tag_strs)
    return {(s, e, t) for s, e, t, _ in spans}


def span_level_f1(gold_spans: set, pred_spans: set) -> dict:
    """
    Compute precision, recall, F1 at span level.

    Args:
        gold_spans: set of (start, end, type) tuples
        pred_spans: set of (start, end, type) tuples

    Returns:
        dict with 'precision', 'recall', 'f1'
    """
    tp = len(gold_spans & pred_spans)
    fp = len(pred_spans - gold_spans)
    fn = len(gold_spans - pred_spans)
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    return {'precision': precision, 'recall': recall, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}


def token_level_f1(gold_ids: np.ndarray, pred_ids: np.ndarray,
                   ignore_id: int = -100) -> float:
    """F1 at the token level, ignoring padding."""
    mask = gold_ids != ignore_id
    correct = (gold_ids[mask] == pred_ids[mask]).sum()
    return float(correct) / (mask.sum() + 1e-8)


# Evaluate greedy vs viterbi on all test sentences
results = []
for i in range(len(X_ner_te)):
    actual_len = (y_ner_te[i] != -100).sum().item()
    gold_ids   = y_ner_te[i].numpy()

    greedy_ids  = np.array(decode_sequence(bilstm_ner, X_ner_te[i], False))
    viterbi_ids = np.array(decode_sequence(bilstm_ner, X_ner_te[i], True, transition))

    gold_spans_i    = extract_spans_from_ids(gold_ids[:actual_len].tolist(), actual_len)
    greedy_spans_i  = extract_spans_from_ids(greedy_ids[:actual_len].tolist(), actual_len)
    viterbi_spans_i = extract_spans_from_ids(viterbi_ids[:actual_len].tolist(), actual_len)

    results.append({
        'tok_greedy':  token_level_f1(gold_ids, greedy_ids),
        'tok_viterbi': token_level_f1(gold_ids, viterbi_ids),
        'span_greedy':  span_level_f1(gold_spans_i, greedy_spans_i)['f1'],
        'span_viterbi': span_level_f1(gold_spans_i, viterbi_spans_i)['f1'],
    })

mean = lambda k: float(np.mean([r[k] for r in results]))
print(f"Evaluation on {len(results)} test sentences:")
print(f"  Greedy  — token F1: {mean('tok_greedy'):.3f}  |  span F1: {mean('span_greedy'):.3f}")
print(f"  Viterbi — token F1: {mean('tok_viterbi'):.3f}  |  span F1: {mean('span_viterbi'):.3f}")
print()
print("Note: span F1 is lower than token F1 because partial matches score zero.")
print("Viterbi improves span F1 by eliminating invalid boundary transitions.")

# Visualise
fig, ax = plt.subplots(figsize=(6, 3))
metrics = ['Token F1\n(Greedy)', 'Token F1\n(Viterbi)',
           'Span F1\n(Greedy)', 'Span F1\n(Viterbi)']
values  = [mean('tok_greedy'), mean('tok_viterbi'),
           mean('span_greedy'), mean('span_viterbi')]
colors  = ['steelblue', 'navy', 'darkorange', 'firebrick']
ax.bar(metrics, values, color=colors)
ax.set_ylim(0, 1)
ax.set_ylabel('F1 Score')
ax.set_title('Token-level vs Span-level F1: Greedy vs Viterbi')
for i, v in enumerate(values):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/07_rw2_f1.png', dpi=100)
plt.close()
print("Plot saved to /tmp/07_rw2_f1.png")

## Real-World Example 3: Entity Confusion Analysis

When a model confuses entity types (predicting ORG when the gold label is PER)
we need to understand where errors cluster. An **entity-level confusion matrix**
shows which entity types are commonly confused with which others.

## Comparison: Token-level vs Span-level F1 and Greedy vs Viterbi

| Metric | Greedy | Viterbi | Difference |
|---|---|---|---|
| Token-level F1 | See above | See above | Viterbi has slight edge |
| Span-level F1 | See above | See above | Larger gap — Viterbi enforces boundaries |
| Invalid BIO transitions | Non-zero | Zero | Viterbi eliminates all |

Key insight: token-level and span-level F1 can diverge significantly because
a single off-by-one boundary error costs 1 span but only 2 tokens.

In [ ]:
# Real-World Example 3: Entity confusion analysis + comparison

ENTITY_TYPES = ['PER', 'ORG', 'LOC']

def entity_confusion_matrix(model: BiLSTMNER,
                             X: torch.Tensor, y: torch.Tensor,
                             use_viterbi: bool = True) -> np.ndarray:
    """
    Build an entity-type confusion matrix.
    Rows = gold type, Cols = predicted type.
    Entities that are missed (FN) are recorded in a special 'MISS' column.

    Returns:
        confusion: [3, 4] matrix (PER/ORG/LOC x PER/ORG/LOC/MISS)
    """
    # 3 types + 1 'MISS' column for false negatives
    cm = np.zeros((3, 4), dtype=int)
    type_to_idx = {'PER': 0, 'ORG': 1, 'LOC': 2}

    for i in range(len(X)):
        actual_len = (y[i] != -100).sum().item()
        gold_ids   = y[i].numpy()[:actual_len].tolist()
        if use_viterbi:
            pred_ids = decode_sequence(model, X[i], True, transition)[:actual_len]
        else:
            pred_ids = decode_sequence(model, X[i], False)[:actual_len]

        gold_spans_i = extract_spans_from_ids(gold_ids, actual_len)
        pred_spans_i = extract_spans_from_ids(pred_ids, actual_len)

        for g_start, g_end, g_type in gold_spans_i:
            gold_r = type_to_idx[g_type]
            # Find best matching prediction (same span boundaries)
            matched = False
            for p_start, p_end, p_type in pred_spans_i:
                if p_start == g_start and p_end == g_end:
                    pred_c = type_to_idx.get(p_type, 3)
                    cm[gold_r, pred_c] += 1
                    matched = True
                    break
            if not matched:
                cm[gold_r, 3] += 1   # missed entirely

    return cm

cm_viterbi = entity_confusion_matrix(bilstm_ner, X_ner_te, y_ner_te, use_viterbi=True)
cm_greedy  = entity_confusion_matrix(bilstm_ner, X_ner_te, y_ner_te, use_viterbi=False)

col_labels = ['PER', 'ORG', 'LOC', 'MISS']
print("Entity confusion matrix (Viterbi) — rows=gold, cols=predicted:")
print(f"{'':>6}  " + "  ".join(f"{c:>6}" for c in col_labels))
for i, etype in enumerate(ENTITY_TYPES):
    print(f"{etype:>6}  " + "  ".join(f"{cm_viterbi[i,j]:>6}" for j in range(4)))

# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cm, title in zip(axes,
                          [cm_greedy, cm_viterbi],
                          ['Greedy Decoding', 'Viterbi Decoding']):
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(4)); ax.set_xticklabels(col_labels)
    ax.set_yticks(range(3)); ax.set_yticklabels(ENTITY_TYPES)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Gold')
    ax.set_title(f'Entity Confusion: {title}')
    for r in range(3):
        for c in range(4):
            ax.text(c, r, str(cm[r, c]), ha='center', va='center', fontsize=10,
                    color='white' if cm[r, c] > cm.max() * 0.6 else 'black')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('/tmp/07_rw3_confusion.png', dpi=100)
plt.close()
print("Entity confusion heatmap saved to /tmp/07_rw3_confusion.png")
print()

# Summary comparison
for label, cm in [('Greedy', cm_greedy), ('Viterbi', cm_viterbi)]:
    tp_total = cm[:3, :3].diagonal().sum()   # correctly typed entities
    total    = cm.sum()
    cross_errors = cm[:3, :3].sum() - tp_total  # right position, wrong type
    missed = cm[:, 3].sum()
    print(f"{label:10s}: correct_type={tp_total}  cross_type_errors={cross_errors}  missed={missed}")